# Method 1 - Part 4: five-fold cross-validation of UNet fusion

On fold I alone (496 test patches), differences between the fusion variants stay within roughly 1 mIoU point and vary across seeds. To evaluate them on the full dataset, we train the five neural variants on folds II to V (seed 0, 100% labels, `32-64-128` UNet, 3,600 steps, `configs/common/`), reuse fold I seed 0 from Part 3, and pool predictions across all five disjoint test splits so all 2,433 patches are evaluated out-of-sample.

| Variant | Description |
|---|---|
| `base_unet-t19-lr1e3-nocw-w32` | TESSERA 2019 alone (128 ch.) |
| `base_unet-ae19-lr1e3-nocw-w32` | AlphaEarth 2019 alone (64 ch.) |
| `m1_unet-t19-ae19-lr1e3-nocw-w32` | Early concatenation of TESSERA 2019 + AlphaEarth 2019 (192 ch.) |
| `m1_unet-t19-noise-lr1e3-nocw-w32` | TESSERA 2019 + 64 N(0, 1) noise channels (dimensionality control) |
| `m1_gated-t19-ae19-lr1e3-nocw-w32` | Pixel-wise gated fusion `a * f1(T) + (1 - a) * f2(AE)` before the UNet |
| `m1_late-t19-ae19-lr1e3-nocw` | Late fusion (mean logits of the trained TESSERA and AlphaEarth UNets) |

We also include gated fusion, since with the `-lr1e3-nocw` recipe its fold I difference against concatenation is -0.38 mIoU points (95% CI [-0.82, +0.03]), compared with -0.61 (95% CI [-1.07, -0.16]) under the initial setup.

## 1. Environment and paths

In [ ]:
import json, os, shutil, platform, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
print("Python", platform.python_version(), "| torch", torch.__version__, f"| {os.cpu_count()} CPUs")
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU, then run again"
print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"  # or the branch being tested
CODE = Path("/content/code")
PROC_DRIVE = DRIVE / "processed"
PROC = Path("/content/processed")
# runs and results; an account that reaches pj_vmi through a shortcut to a folder shared by another
# account cannot create folders in it: set WORK to a folder of its own, e.g. Path("/content/drive/MyDrive/pj_vmi_runs_alt")
WORK = DRIVE
RUNS, RUNS_CSV = WORK / "runs", WORK / "results" / "runs.csv"
OUT = WORK / "results" / "m1"
CONFIGS = ["configs/common/b2_unet_t19.yaml", "configs/common/b5_unet_ae19.yaml",
           "configs/common/m1_1_concat_t19_ae19.yaml", "configs/common/m1_2_noise_t19.yaml",
           "configs/common/m1_4_gated_t19_ae19.yaml"]
T, AE = "base_unet-t19-lr1e3-nocw-w32", "base_unet-ae19-lr1e3-nocw-w32"
CONCAT, NOISE, GATED = "m1_unet-t19-ae19-lr1e3-nocw-w32", "m1_unet-t19-noise-lr1e3-nocw-w32", "m1_gated-t19-ae19-lr1e3-nocw-w32"
LATE_VARIANT = "late-t19-ae19-lr1e3-nocw"
LATE = f"m1_{LATE_VARIANT}"

missing = [v for v in (T, AE, CONCAT, NOISE, GATED, LATE) if not (RUNS / f"{v}_100_foldI_s0" / "run.json").exists()]
assert not missing, f"fold I seed 0 missing for {missing}: run notebooks/m1/03_fusion_unet.ipynb first"
print("fold I seed 0 found for the 6 variants")

## 2. Repository and unit tests

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt
!python -m pytest -q tests
sys.path.insert(0, str(CODE))

## 3. Copy TESSERA 2019 and AlphaEarth 2019 to local disk

In [ ]:
PROC.mkdir(parents=True, exist_ok=True)
for f in ["meta.csv", "splits.json", "labels.npy", "instances.npy", "valid_mask.npy",
          "tessera_2019_q.npy", "tessera_2019_scale.npy", "ae_2019.npy"]:
    if not (PROC / f).exists() or (PROC / f).stat().st_size != (PROC_DRIVE / f).stat().st_size:
        shutil.copy(PROC_DRIVE / f, PROC / f)
    print(f"{f:24s} {(PROC / f).stat().st_size / 1e9:6.2f} GB  ok={(PROC / f).stat().st_size == (PROC_DRIVE / f).stat().st_size}")

## 4. Fold II
Train the five neural variants (`CONFIGS`) on fold II (seed 0) and evaluate late fusion from the trained TESSERA and AlphaEarth models.

In [ ]:
SPLIT = "II"
for cfg in CONFIGS:
    !python scripts/train_head.py --config {cfg} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --split {SPLIT} --seeds 0
!python scripts/m1_late_fusion.py --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --a {T} --b {AE} --variant {LATE_VARIANT} --split {SPLIT} --seeds 0

## 5. Fold III

In [ ]:
SPLIT = "III"
for cfg in CONFIGS:
    !python scripts/train_head.py --config {cfg} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --split {SPLIT} --seeds 0
!python scripts/m1_late_fusion.py --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --a {T} --b {AE} --variant {LATE_VARIANT} --split {SPLIT} --seeds 0

## 6. Fold IV

In [ ]:
SPLIT = "IV"
for cfg in CONFIGS:
    !python scripts/train_head.py --config {cfg} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --split {SPLIT} --seeds 0
!python scripts/m1_late_fusion.py --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --a {T} --b {AE} --variant {LATE_VARIANT} --split {SPLIT} --seeds 0

## 7. Fold V

In [ ]:
SPLIT = "V"
for cfg in CONFIGS:
    !python scripts/train_head.py --config {cfg} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --split {SPLIT} --seeds 0
!python scripts/m1_late_fusion.py --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --a {T} --b {AE} --variant {LATE_VARIANT} --split {SPLIT} --seeds 0

## 8. Five-fold pooled evaluation
Pool the per-patch confusion matrices across folds I to V (all 2,433 patches) to compute overall mIoU, boundary/core mIoU, per-fold breakdowns, and paired bootstrap differences.

In [ ]:
from IPython.display import Image, display
from scripts.summarize_folds import summarize_folds

pairs = [f"{CONCAT}:{T}", f"{CONCAT}:{AE}", f"{CONCAT}:{NOISE}", f"{NOISE}:{T}", f"{GATED}:{T}", f"{GATED}:{CONCAT}",
         f"{LATE}:{T}", f"{LATE}:{CONCAT}"]
pd.set_option("display.width", 220)
per_fold, summary, deltas = summarize_folds(RUNS, OUT, name="fusion_unet_5folds-lr1e3-nocw", variants=[T, AE, CONCAT, NOISE, GATED, LATE],
                                            pairs=pairs, seed=0)
print(per_fold.pivot(index="variant", columns="split", values="miou")[["I", "II", "III", "IV", "V"]].round(4).to_string())
print("\npooled over the five folds:")
print(summary[["variant", "patches", "miou_pooled", "miou_ci_low", "miou_ci_high", "boundary_pooled", "core_pooled",
               "miou_fold_mean", "miou_fold_std", "n_params"]].round(4).to_string(index=False))
print("\npaired bootstrap on the pooled patches, delta = mIoU(variant) - mIoU(versus):")
print(deltas.round(4).to_string(index=False))
display(Image(str(OUT / "fusion_unet_5folds-lr1e3-nocw_per_class.png")))